<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/lab4_with_pipieline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

!wget -q -O bank.csv https://raw.githubusercontent.com/selva86/datasets/master/bank-full.csv


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("BankMarketing_With_Pipeline").getOrCreate()


In [ ]:
from pyspark.sql.functions import col

df = spark.read.format("csv").option("header", "true").option("sep", ";").option("inferSchema", "true").load("bank.csv")

dataset = df.select(
    col("age").cast("float"),
    col("job"),
    col("marital"),
    col("education"),
    col("duration").cast("float"),
    col("campaign").cast("float"),
    col("y"),
).dropna(how="any")

(train_df, test_df) = dataset.randomSplit([0.8, 0.2], seed=42)
print("Training Dataset Count: ", train_df.count())
print("Test Dataset Count: ", test_df.count())


Training Dataset Count:  32977
Test Dataset Count:  8211


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

job_indexer = StringIndexer(inputCol="job", outputCol="job_index", handleInvalid="keep")
marital_indexer = StringIndexer(inputCol="marital", outputCol="marital_index", handleInvalid="keep")
education_indexer = StringIndexer(inputCol="education", outputCol="education_index", handleInvalid="keep")
label_indexer = StringIndexer(inputCol="y", outputCol="label")

feature_cols = ["age", "duration", "campaign", "job_index", "marital_index", "education_index"]
vector_assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

rf_classifier = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20, maxDepth=5)

pipeline = Pipeline(stages=[
    job_indexer,
    marital_indexer,
    education_indexer,
    label_indexer,
    vector_assembler,
    rf_classifier
])



In [ ]:
pipeline_model = pipeline.fit(train_df)

train_predictions = pipeline_model.transform(train_df)
test_predictions = pipeline_model.transform(test_df)

test_predictions.select("y", "label", "features", "prediction", "prediction").show(5, truncate=False)

#

+---+-----+-----------------------------+----------+----------+
|y  |label|features                     |prediction|prediction|
+---+-----+-----------------------------+----------+----------+
|no |0.0  |[18.0,75.0,1.0,10.0,1.0,1.0] |0.0       |0.0       |
|no |0.0  |[19.0,43.0,3.0,10.0,1.0,2.0] |0.0       |0.0       |
|yes|1.0  |[19.0,126.0,4.0,10.0,1.0,2.0]|0.0       |0.0       |
|no |0.0  |[19.0,72.0,4.0,10.0,1.0,6.0] |0.0       |0.0       |
|yes|1.0  |[19.0,205.0,1.0,10.0,1.0,6.0]|0.0       |0.0       |
+---+-----+-----------------------------+----------+----------+
only showing top 5 rows


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
train_acc = evaluator_acc.evaluate(train_predictions)
test_acc = evaluator_acc.evaluate(test_predictions)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1")
train_f1 = evaluator_f1.evaluate(train_predictions)
test_f1 = evaluator_f1.evaluate(test_predictions)

print("=== Results with Pipeline ===")
print("Training Accuracy: ", train_acc)
print("Test Accuracy: ", test_acc)
print("Training F1-Score: ", train_f1)
print("Test F1-Score: ", test_f1)


=== Results with Pipeline ===
Training Accuracy:  0.8934712072050217
Test Accuracy:  0.8894166362197052
Training F1-Score:  0.8557285273380532
Test F1-Score:  0.8498865871648326
